### Bronze to Silver Transformation

## Worker notebook

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("folder_name","","Folder Name")
dbutils.widgets.text("dataset_name","","Dataset Name")
dataset_name=dbutils.widgets.get("dataset_name")
folder_name=dbutils.widgets.get("folder_name")

In [0]:
bronze_path = f"abfss://bronze@aze2estorage.dfs.core.windows.net/{folder_name}/{dataset_name}"
silver_path = f"abfss://silver@aze2estorage.dfs.core.windows.net/{folder_name}/"

In [0]:

spark.conf.set("fs.azure.account.key.aze2estorage.dfs.core.windows.net", dbutils.secrets.get(scope="storageaccess_for_e2e", key="app-secret"))



In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

In [0]:
def transform_calender(df):
    return(

        df.withColumn("year",year(col("Date")))\
        .withColumn("month",month(col("Date")))\
        .withColumn("quarter",quarter(col("date")))\
        .withColumn("day_of_week",dayofweek(col("date")))
    )

In [0]:
def transform_customers(df):
    return(
        df.withColumnRenamed("CustomerKey", "customer_key")\
    .withColumnRenamed("Prefix", "prefix")\
    .withColumnRenamed("FirstName", "first_name")\
    .withColumnRenamed("LastName", "last_name")\
    .withColumnRenamed("BirthDate", "birth_date")\
    .withColumnRenamed("MaritalStatus", "marital_status")\
    .withColumnRenamed("Gender", "gender")\
    .withColumnRenamed("EmailAddress", "email_address")\
    .withColumnRenamed("AnnualIncome", "annual_income")\
    .withColumnRenamed("TotalChildren", "total_children")\
    .withColumnRenamed("EducationLevel", "education_level")\
    .withColumnRenamed("Occupation", "occupation")\
    .withColumnRenamed("HomeOwner", "home_owner")\
    .withColumn("annual_income", regexp_replace(col("annual_income"), "[$,]", "").cast(DecimalType(12,2)))\
    .withColumn("full_name",trim(concat_ws(" ",col("prefix"),col("first_name"),col("last_name"))))\
    .withColumn(
        "marital_status",
        when(col("marital_status") == "M", "Married")
        .when(col("marital_status") == "S", "Single")
        .otherwise("Unknown")
    )
    .withColumn(
        "gender",
        when(col("gender") == "M", "Male")
        .when(col("gender") == "F", "Female")
        .otherwise("Unknown")
    )
    .withColumn(
        "is_home_owner",
        when(col("home_owner") == "Y", True)
        .when(col("home_owner") == "N", False)
        .otherwise(None)
    )
    .withColumn("email_address", lower(trim(col("email_address"))))\
    .withColumn("age", floor(datediff(current_date(), col("birth_date")) / 365.25))
    .withColumn(
        "age_group",
        when(col("age") < 25, "<25")
        .when((col("age") >= 25) & (col("age") < 40), "25-39")
        .when((col("age") >= 40) & (col("age") < 60), "40-59")
        .otherwise("60+")
    )
    .withColumn("ingested_at", current_timestamp())\
    .select(
        "customer_key",
        "full_name",
        "prefix",
        "first_name",
        "last_name",
        "birth_date",
        "age",
        "age_group",
        "marital_status",
        "gender",
        "email_address",
        "annual_income",
        "total_children",
        "education_level",
        "occupation",
        "is_home_owner",
        "ingested_at"
    )
    )

In [0]:
def transform_orders(df):
    return(
        df.withColumnRenamed("OrderDate", "order_date")\
        .withColumnRenamed("StockDate", "stock_date")\
        .withColumnRenamed("OrderNumber", "order_number")\
        .withColumnRenamed("ProductKey", "product_key")\
        .withColumnRenamed("CustomerKey", "customer_key")\
        .withColumnRenamed("TerritoryKey", "territory_key")\
        .withColumnRenamed("OrderLineItem", "order_line_item")\
        .withColumnRenamed("OrderQuantity", "order_quantity")\
        .withColumn("days_to_stock", datediff(col("stock_date"), col("order_date")))\
        .withColumn("order_year", year(col("order_date")))\
        .withColumn("order_month", month(col("order_date")))\
        .withColumn("ingested_at", current_timestamp())\
        .withColumn(
            "row_num",
            row_number().over(
                Window.partitionBy("order_number", "order_line_item").orderBy(col("ingested_at").desc())
            )
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
        .drop("days_to_stock")
    )
        
        

In [0]:
def transform_product_categories(df):
    return(
        df.withColumnRenamed("ProductCategoryKey", "product_category_key")\
        .withColumnRenamed("CategoryName", "category_name")\
        .withColumn("ingested_at", current_timestamp())\
        .select(
            "product_category_key",
            "category_name",
            "ingested_at"
        )
    )


In [0]:
def transform_product_sub_categories(df):
    return(
        df.withColumnRenamed("ProductSubcategoryKey", "product_subcategory_key")\
        .withColumnRenamed("SubcategoryName", "subcategory_name")\
        .withColumnRenamed("ProductCategoryKey", "product_category_key")\
        .withColumn("ingested_at", current_timestamp())\
        .select(
            "product_subcategory_key",
            "subcategory_name",
            "product_category_key",
            "ingested_at"
        )
    )

In [0]:
def transform_products(df):
    return (
        df.withColumnRenamed("ProductKey", "product_key")
        .withColumnRenamed("ProductSubcategoryKey", "product_subcategory_key")
        .withColumnRenamed("ProductSKU", "product_sku")
        .withColumnRenamed("ProductName", "product_name")
        .withColumnRenamed("ModelName", "model_name")
        .withColumnRenamed("ProductDescription", "product_description")
        .withColumnRenamed("ProductColor", "color")
        .withColumnRenamed("ProductSize", "size")
        .withColumnRenamed("ProductStyle", "product_style")
        .withColumnRenamed("ProductCost", "product_cost")
        .withColumnRenamed("ProductPrice", "product_price")
        .withColumn("product_cost", round(col("product_cost").cast("double"), 2))
        .withColumn("product_price", round(col("product_price").cast("double"), 2))
        .withColumn("ingested_at", current_timestamp())
        .withColumn("profit_margin_amount", round(col("product_price") - col("product_cost"), 2))
        .withColumn("profit_margin_pct", round((col("profit_margin_amount") / col("product_price")) * 100, 2))
        .withColumn(
            "row_num",
            row_number().over(
                Window.partitionBy("product_key").orderBy(col("ingested_at").desc())
            )
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

In [0]:
def transform_returns(df):
    return(
        df.withColumnRenamed("ReturnDate", "return_date")\
        .withColumnRenamed("TerritoryKey", "territory_key")\
        .withColumnRenamed("ProductKey", "product_key")\
        .withColumnRenamed("ReturnQuantity", "return_quantity")\
        .withColumn("ingested_at", current_timestamp())\
        .select(
            "return_date",
            "territory_key",
            "product_key",
            "return_quantity",
            "ingested_at"
        )
    )

In [0]:
def transform_territories(df):
    return (
        df
        # 1. Column Renamings
        .withColumnRenamed("SalesTerritoryKey", "sales_territory_key")
        .withColumnRenamed("Region", "region")
        .withColumnRenamed("Country", "country")
        .withColumnRenamed("Continent", "continent")
        
        # 2. String Cleaning
        .withColumn("region", trim(col("region")))
        .withColumn("country", trim(col("country")))
        .withColumn("continent", trim(col("continent")))
        
        # 3. Data Type Casting
        .withColumn("sales_territory_key", col("sales_territory_key").cast(LongType()))
        
        # 4. Audit Timestamp
        .withColumn("ingested_at", current_timestamp())
        
        # 5. Deduplication on Primary Key
        .withColumn(
            "row_num",
            row_number().over(
                Window.partitionBy("sales_territory_key").orderBy(col("ingested_at").desc())
            )
        )
        .filter(col("row_num") == 1)
        .drop("row_num")
    )

In [0]:
TRANSFORM_MAP = {
    "AdventureWorks_Calendar.csv": transform_calender,
    "AdventureWorks_Customers.csv": transform_customers,
    "AdventureWorks_Product_Categories.csv": transform_product_categories,
    "AdventureWorks_Product_Subcategories.csv": transform_product_sub_categories,
    "AdventureWorks_Products.csv": transform_products,
    "AdventureWorks_Returns.csv": transform_returns,
    "AdventureWorks_Sales_2015.csv": transform_orders,
    "AdventureWorks_Sales_2016.csv": transform_orders,
    "AdventureWorks_Sales_2017.csv": transform_orders,
    "AdventureWorks_Territories.csv": transform_territories,
}

In [0]:
def default_transform(df):
    print(f"⚠️ Warning: No specific transform found for '{dataset_name}'. Applying default identity pass-through.")
    return df

In [0]:
transformation_fn = TRANSFORM_MAP.get(dataset_name,default_transform)

In [0]:
df_silver_transformed = transformation_fn(df)

In [0]:
df_silver_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(silver_path)